# TXT Approach

In [ ]:
OUTPUT = '../data/output.txt'

In [ ]:
text = []

with open(OUTPUT, "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        if line != '\'':
            text.append(line + '\n')

# list(product([0], [0, 1, 2, 3, 4, 5])) + list(product([1,2], [0,1,2,3,4]))
for lap, segment in list(product([0], [0, 1, 2, 3, 4])):
    print(f'Lap: {lap}, Segment: {segment}')
    
    # Filter text for lap and segment
    filterd_lines = [row for row in text if f"'lap_number': {lap}" in row and f"'segment': {segment}" in row]
    filtered_text = '\n'.join(filterd_lines)   
    
    # # --- RAG setup --- TODO
    # retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    # question = "How do I setup my car best in Forza Horizon 5?"
    # retrieved_docs = retriever.invoke(question)
    # retrieved_docs[2].page_content
    #rag_context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Generate LLM text from output file
    resp = ollama.chat(
        model="nemotron-3-nano:30b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + f"\n```json\n{filtered_text}\n```"},
        ],
    )

    # Loggging
    data_tokens = count_tokens(filtered_text, model_name="gpt-4")
    print(f"Token-Anzahl User {data_tokens}")
    timestamps = []
    for line in filterd_lines:
        num = r"[-+]?(?:\d*\.\d+|\d+\.?\d*)(?:[eE][-+]?\d+)?"
        pattern = rf"timestamp in s'\s*:\s*({num})\s*,\s*({num})"
        timestamps += [float(re.search(pattern, line).group(1))]
    log_response(timestamps, lap, segment, system_prompt, user_prompt, resp)